In [1]:
%pip install -q rank-bm25 sentence-transformers langchain-google-genai pandas numpy python-dotenv

import os, getpass
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [2]:
corpus = [
    "Transformers use self-attention mechanisms to capture relationships between tokens in a sequence.",
    "Self-attention computes pairwise interactions between tokens to determine contextual importance.",
    "Positional encoding injects order information into transformer embeddings.",

    "Gradient descent updates model weights using computed gradients.",
    "Training neural networks involves optimization algorithms like Adam, SGD, and learning rate scheduling.",
    "Regularization methods such as dropout and weight decay reduce overfitting.",

    "BM25 is a keyword-based retrieval algorithm using term frequency and inverse document frequency.",
    "Sparse retrieval relies on exact keyword matching rather than semantic similarity.",
    "Dense retrieval uses vector embeddings to find semantically similar documents.",

    "Query expansion techniques reformulate user queries to address vocabulary mismatch in retrieval systems."
]

doc_ids = [f"doc_{i}" for i in range(len(corpus))]

In [3]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

corpus_emb = embed_model.encode(corpus, normalize_embeddings=True)

def dense_retrieve(query, k=1):
    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(corpus_emb, q_emb)
    idx = np.argsort(scores)[::-1][:k]
    return [{"doc_id": doc_ids[i], "text": corpus[i]} for i in idx]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)

        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.emb = self.sbert.encode(corpus, normalize_embeddings=True)

    def retrieve(self, query, top_k=8):
        # BM25
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_rank = np.argsort(bm25_scores)[::-1]
        bm25_map = {idx: i+1 for i, idx in enumerate(bm25_rank)}

        # SBERT
        q_emb = self.sbert.encode([query], normalize_embeddings=True)[0]
        sbert_scores = np.dot(self.emb, q_emb)
        sbert_rank = np.argsort(sbert_scores)[::-1]
        sbert_map = {idx: i+1 for i, idx in enumerate(sbert_rank)}

        # RRF fusion
        results = []
        for i in range(len(self.corpus)):
            rrf = (1/(self.k + bm25_map[i])) + (1/(self.k + sbert_map[i]))
            results.append({
                "doc_id": f"doc_{i}",
                "rrf_score": rrf,
                "bm25_rank": bm25_map[i],
                "sbert_rank": sbert_map[i],
                "text": self.corpus[i]
            })

        return sorted(results, key=lambda x: x["rrf_score"], reverse=True)[:top_k]

retriever = HybridRetriever(corpus)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def multi_query(query):
    prompt = f"""
Generate 3 different improved search queries for this AI/ML question.
Make them more detailed and technical.

Question: {query}

Return exactly 3 queries, one per line.
"""
    output = llm.invoke(prompt).content.strip().split("\n")
    return list(set([q.strip("- ").strip() for q in output if q.strip()]))


def rerank(query, docs, top_k=3):
    pairs = [[query, d["text"]] for d in docs]
    scores = cross_encoder.predict(pairs)

    for d, s in zip(docs, scores):
        d["cross_score"] = float(s)

    return sorted(docs, key=lambda x: x["cross_score"], reverse=True)[:top_k]


def generate(query, docs):
    context = "\n".join([d["text"] for d in docs])
    prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question: {query}
"""
    return llm.invoke(prompt).content

In [6]:
def advanced_rag(user_query: str) -> str:
    queries = multi_query(user_query)

    all_docs = []
    for q in queries:
        all_docs.extend(retriever.retrieve(q, 8))

    # Deduplicate
    unique_docs = {d["text"]: d for d in all_docs}.values()

    reranked = rerank(user_query, list(unique_docs), 3)

    return generate(user_query, reranked)

In [7]:
def naive_top(query):
    return dense_retrieve(query, 1)[0]


def advanced_top(query):
    queries = multi_query(query)

    all_docs = []
    for q in queries:
        all_docs.extend(retriever.retrieve(q, 8))

    unique_docs = {d["text"]: d for d in all_docs}.values()

    return rerank(query, list(unique_docs), 1)[0]

In [8]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how can retrieval handle vocabulary mismatch?"
]

for q in queries:
    naive = naive_top(q)
    advanced = advanced_top(q)

    print("=" * 100)
    print("QUERY:", q)

    print("\nNaïve RAG Top Doc:")
    print(naive["text"])

    print("\nAdvanced RAG Top Doc:")
    print(advanced["text"])

    print("\nDifferent?:", "YES" if naive["text"] != advanced["text"] else "NO")
    print("=" * 100, "\n")

QUERY: how do transformers encode meaning?

Naïve RAG Top Doc:
Positional encoding injects order information into transformer embeddings.

Advanced RAG Top Doc:
Transformers use self-attention mechanisms to capture relationships between tokens in a sequence.

Different?: YES

QUERY: optimization techniques for training

Naïve RAG Top Doc:
Training neural networks involves optimization algorithms like Adam, SGD, and learning rate scheduling.

Advanced RAG Top Doc:
Training neural networks involves optimization algorithms like Adam, SGD, and learning rate scheduling.

Different?: NO

QUERY: how can retrieval handle vocabulary mismatch?

Naïve RAG Top Doc:
Query expansion techniques reformulate user queries to address vocabulary mismatch in retrieval systems.

Advanced RAG Top Doc:
Query expansion techniques reformulate user queries to address vocabulary mismatch in retrieval systems.

Different?: NO



## Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Positional encoding injects order information into transformer embeddings. | Transformers use self-attention mechanisms to capture relationships between tokens in a sequence. | Yes |
| optimization techniques for training | Training neural networks involves optimization algorithms like Adam, SGD, and learning rate scheduling. | Training neural networks involves optimization algorithms like Adam, SGD, and learning rate scheduling. | No |
| how can retrieval handle vocabulary mismatch? | Query expansion techniques reformulate user queries to address vocabulary mismatch in retrieval systems. | Query expansion techniques reformulate user queries to address vocabulary mismatch in retrieval systems. | No |

### Observation

The advanced RAG pipeline improves retrieval quality for ambiguous queries by leveraging query expansion, hybrid retrieval, and re-ranking.

For the transformer-related query, the naïve system retrieves a less relevant document, while the advanced system correctly identifies the self-attention mechanism, demonstrating improved understanding.

However, for queries with clear and direct keywords, both naïve and advanced systems retrieve the same document, indicating that dense retrieval alone can be sufficient in such cases.

This shows that advanced RAG is particularly beneficial in handling vague or vocabulary-mismatched queries.